In [0]:
try:
    API_KEY = dbutils.widgets.get("API_KEY")
except Exception as e:
    raise Exception("Please set the API_KEY widget")

In [0]:
import requests

url = (
    "https://www.alphavantage.co/query"
    "?function=GOLD_SILVER_SPOT"
    "&symbol=GOLD"
    f"&apikey={API_KEY}"
)

response = requests.get(url)
data = response.json()


In [0]:
df = spark.createDataFrame([data])

In [0]:
def CRUD_Delta(path):
    file_path = f"/Volumes/workspace/default/invest/{path}"

    (df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "True")
    .save(file_path)
    )

    return file_path

In [0]:
def CRUD(df, path):
    file_path = CRUD_Delta(path)
    
    spark.sql("use catalog workspace")
    spark.sql("use schema default") 

    if spark.catalog.tableExists(f"workspace.default.{path}"):
        spark.sql(f"""
            INSERT INTO workspace.default.{path}
            SELECT * FROM delta.`{file_path}`
        """)
    else:
        spark.sql(f"""
            CREATE TABLE workspace.default.{path}
            AS SELECT * FROM delta.`{file_path}`
        """)

In [0]:
CRUD(df, "bronze_xauusd")